In [31]:
%load_ext autoreload
%autoreload 2

import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
from datetime import timedelta

from analyse.advertising.e00_download.mediatree import (
    CachedMediatreeAPI,
    all_intervals_between,
)
from analyse.advertising.e02_finger_printer.finger_printer import (
    FingerprintPipeline,
    print_report,
    plot_groups,
)

from datetime import datetime
from zoneinfo import ZoneInfo
import json
from pathlib import Path

tz_paris = ZoneInfo("Europe/Paris")
channel = "tf1"
output_prefix = "PANZANI"

MONDAY_LAST_YEAR = datetime(2025, 3, 10, tzinfo=tz_paris)

SIMILARITY_THRESHOLD = 0.2

OUTPUT_FILE = f"working_data/{output_prefix}_report_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.json"
OUTPUT_PLOT = f"working_data/{output_prefix}_plot_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.png"


In [33]:
api = CachedMediatreeAPI()

INPUT_SEGMENTS_FOLDER = os.path.join(
    "../working_data", f"segments_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}"
)

input_files = []

CUSTOM_DATES = [
    (
        datetime(2025, 3, 10, 12, 0, tzinfo=tz_paris),
        datetime(2025, 3, 10, 12, 30, tzinfo=tz_paris),
    ),
    (
        datetime(2025, 3, 11, 12, 0, tzinfo=tz_paris),
        datetime(2025, 3, 11, 12, 30, tzinfo=tz_paris),
    ),
    (
        datetime(2025, 3, 11, 12, 30, tzinfo=tz_paris),
        datetime(2025, 3, 11, 13, 0, tzinfo=tz_paris),
    ),
]

for start_date, end_date in CUSTOM_DATES:
    audio_file = await api.download_export(
        channel, start_date, end_date + timedelta(minutes=1)
    )  # on ajoute une minute pour être sûr de couvrir toute la période

    segment_file = os.path.join(
        INPUT_SEGMENTS_FOLDER,
        f"{start_date.strftime('%Y-%m-%d_%H-%M-%S')}.json",
    )
    input_files.append((audio_file, segment_file, start_date.timestamp()))

input_files


[('../exports/tf1_2025-03-10_11-00-00Z_2025-03-10_11-31-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_12-00-00.json',
  1741604400.0),
 ('../exports/tf1_2025-03-11_11-00-00Z_2025-03-11_11-31-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-11_12-00-00.json',
  1741690800.0),
 ('../exports/tf1_2025-03-11_11-30-00Z_2025-03-11_12-01-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-11_12-30-00.json',
  1741692600.0)]

In [34]:
pipeline = FingerprintPipeline(
    sources=input_files,
    similarity_threshold=0.2,
)

if False:
    fingerprints = pipeline.fingerprints()
    pipeline.diagnose(fingerprints)
else:
    report, fingerprints, groups = pipeline.run()
    print_report(report)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"  Rapport JSON : {Path(OUTPUT_FILE).absolute()}")

    # Durée totale = fin du dernier segment
    total = max(fp.end_sec for fp in fingerprints) if fingerprints else 3600
    plot_groups(report, total, OUTPUT_PLOT)
    print(f"  Graphique : {Path(OUTPUT_PLOT).absolute()}")


  [Cache] Chargement depuis tf1_2025-03-10_11-00-00Z_2025-03-10_11-31-00Z_cc104e11_86a11994.json
Chargement audio : ../exports/tf1_2025-03-11_11-00-00Z_2025-03-11_11-31-00Z.mp3



[Fingerprinting] 305 segments (../exports/tf1_2025-03-11_11-00-00Z_2025-03-11_11-31-00Z.mp3)...


Empreintes: 100%|██████████| 305/305 [00:04<00:00, 74.07it/s] 


  305 empreintes générées
  [Cache] Sauvegardé : tf1_2025-03-11_11-00-00Z_2025-03-11_11-31-00Z_78942cc6_86a11994.json
  [Cache] Chargement depuis tf1_2025-03-11_11-30-00Z_2025-03-11_12-01-00Z_f3d554b2_86a11994.json

[Total] 857 empreintes sur 3 source(s)

[Clustering] 857 segments — construction de l'index inversé...
  205 paires candidates → 9 retenues (≥ 2 hashes communs)  [vs 366796 paires en O(n²)]


Comparaison fingerprints: 100%|██████████| 9/9 [00:00<00:00, 41619.33it/s]


  9 paires similaires trouvées

══════════════════════════════════════════════════════════════════════
  RAPPORT DE RÉPÉTITIONS
══════════════════════════════════════════════════════════════════════
  Segments analysés : 857
  Groupes détectés  : 848

  Classification             Groupes   Occurrences tot
  ────────────────────────────────────────────────────
  SEGMENT_UNIQUE                 839               839
  CONTENU_REPETE                   9                18

  TOP 15 GROUPES LES PLUS FRÉQUENTS :
  Groupe    Occurrences   Durée moy  Classification
  ────────────────────────────────────────────────────────────
  G623                2      20.4s  CONTENU_REPETE
    ↳ [2025-03-10 11:25:14 UTC]  20.2s
    ↳ [2025-03-11 11:30:56 UTC]  20.5s
  G626                2       1.4s  CONTENU_REPETE
    ↳ [2025-03-10 11:28:07 UTC]  1.5s
    ↳ [2025-03-11 11:31:37 UTC]  1.2s
  G628                2       4.0s  CONTENU_REPETE
    ↳ [2025-03-10 11:28:16 UTC]  4.0s
    ↳ [2025-03-11 11:31:46 UT

In [ ]:
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)
from analyse.advertising.tools.testimony_table.extract import get_testimony_data
from analyse.advertising.e00_download.mediatree import CachedMediatreeAPI
from analyse.advertising.tools.basic_elements.segments import (
    load_segment_groups_from_json,
)
from analyse.advertising.e01_rupture_detection.rupture_detector import (
    RuptureDetector,
    print_summary,
)
import httpx

mediatree_api = CachedMediatreeAPI()

TESTIMONY_CHANNEL = "TF1"

detector = RuptureDetector(
    sensitivity=0.1,
    context_sec=1.0,  # durée analysée de chaque côté d'un point pour évaluer la rupture.
    max_ruptures=0,  # 0 pour pas de limite
    sr=22050,  # Fréquence d'échantillonnage de travail
    hop_length=512,  # Pas entre frames (~23ms à 22050Hz)
    n_mfcc=10,  # Nombre de coefficients MFCC
    novelty_smooth_sec=0.5,  # Lissage temporel de la courbe de nouveauté.
    min_segment_sec=1.0,  # Durée minimale d'un segment pour être considéré comme tel.
    silence_percentile=8.0,  # Percentile pour définir le seuil de silence
    cosine_weight=1.0,
)

for start_date, end_date in CUSTOM_DATES:
    annotations = get_testimony_data(
        channel=TESTIMONY_CHANNEL,
        from_date=start_date,
        to_date=end_date,
        source_file="export.csv",
    )

    async with httpx.AsyncClient(timeout=httpx.Timeout(60.0, connect=60.0)) as client:
        witness_file = await mediatree_api.api.get_single_export_url(
            client, channel, start_date, end_date, media_format="mp4"
        )
    input_file = await mediatree_api.download_export(channel, start_date, end_date)
    segments, peaks, novelty, features, y = detector.run(input_file)
    

    output_path = f"working_data/{output_prefix}_result_{channel}_{start_date.strftime('%Y%m%d_%H%M%S')}.html"
    generate_player(
        media_input=witness_file,
        segments=[s.to_dict() for s in segments],
        annotations=format_annotation(annotations, from_date=start_date),
        output_path=output_path,
        start_epoch=int(start_date.timestamp()),
        params=detector.get_params(),
        novelty_peaks=detector.get_novelty_peaks(novelty),
        groups=report["groups"],
    )

[1/5] Chargement : ../exports/tf1_2025-03-10_11-00-00Z_2025-03-10_11-30-00Z.mp3
      Durée : 1800.0s  (30.0 min)
      → 4.17s
[2/5] Extraction des features...
      → 3.87s
[3/5] Calcul de la courbe de nouveauté...
      Fenêtre de contexte : 43 frames = 1.00s de chaque côté
      Seuil silence       : énergie < 0.00834 (percentile 8.0) → 12285 frames silencieuses (15.8%)
      Lissage             : 21 frames = 0.49s
      → 1.61s
[4/5] Détection des frontières...
      Seuil sensitivity=0.10 → hauteur min 0.0073 (percentile 90)
      Après seuil + distance min (1.0s) : 308 ruptures candidates
      Résultat final : 308 ruptures → 309 segments
      → 0.00s
[5/5] Segmentation terminée : 309 segments  → 0.01s
      Durée totale : 9.66s
[1/3] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-10T11-00-00Z_2025-03-10T11-30-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/3] Segments fournis en entrée
  309 segments
[3/3] Annotations fournies en